# 05. Construccion de la base anual de modelado

Este notebook transforma la base mensual integrada en una base anual de modelado a nivel `departamento-anio`.

## Objetivos
- partir de la base mensual integrada ya validada
- conservar la capa anual observada de Agronet
- agregar variables climaticas y satelitales de forma metodologicamente consistente
- construir una version anual del target de modelado
- exportar una base anual lista para EDA y fase formal de modelos

## Resultado esperado
Al finalizar este notebook debemos tener un dataset anual con:

1. una fila por `departamento-anio`
2. variables productivas observadas de Agronet
3. agregados anuales y de cosecha para clima y satelite
4. una definicion operativa del target anual
5. una exportacion lista para continuar con EDA y diseno experimental


## Comentario metodologico

Este notebook formaliza la capa principal de modelado del proyecto. La unidad de analisis es `departamento-anio`, coherente con la disponibilidad observada de Agronet.

Aqui se construyen dos tipos de variables explicativas:

- agregados anuales completos
- agregados de meses de cosecha

Y se construyen tambien dos referencias de rendimiento:

- una referencia departamental de muestra completa, consistente con la logica ya usada en el proyecto
- una referencia historica basada en años previos, util para futuras pruebas metodologicas

La variable objetivo principal que se deja operativa en este notebook sera la perdida porcentual anual respecto a la referencia departamental de muestra completa. Tambien se conservara la version contra referencia historica previa para comparaciones posteriores.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 160)


def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'
INPUT_PATH = BASE_DATOS / 'INTERMEDIAS' / 'base_mensual_integrada.csv'
OUTPUT_DIR = BASE_DATOS / 'FINALES'
OUTPUT_PATH = OUTPUT_DIR / 'dataset_modelado_anual.csv'

print(f'Ruta actual: {CURRENT_DIR}')
print(f'Raiz detectada del proyecto: {PROJECT_ROOT}')
print(f'Archivo de entrada: {INPUT_PATH}')
print(f'Archivo de salida: {OUTPUT_PATH}')


Ruta actual: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\MODELOS
Raiz detectada del proyecto: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2
Archivo de entrada: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\INTERMEDIAS\base_mensual_integrada.csv
Archivo de salida: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\FINALES\dataset_modelado_anual.csv


In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'No existe el archivo esperado: {INPUT_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Rutas validadas correctamente.')


Rutas validadas correctamente.


## Carga de la base mensual integrada

La base mensual integrada es la fuente inmediata para construir la capa anual de modelado.


In [3]:
df = pd.read_csv(INPUT_PATH, sep=';')
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df = df.sort_values(['departamento', 'fecha']).reset_index(drop=True)

print('Shape base mensual integrada:', df.shape)
display(df.head(5))


Shape base mensual integrada: (628, 56)


,fecha,departamento,mes,anio,precipitation,temp_aire_C,humedad_relativa_pct,soil,def,pet,aet,GDD_cafe,NDVI,EVI,LST_Day_1km,LST_Night_1km,Gpp,Lai_500m,Fpar_500m,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio,NDVI_anomalia_pct,EVI_anomalia_pct,Gpp_anomalia_pct,Lai_500m_anomalia_pct,precipitation_anomalia_pct,indice_perdida,evento_perdida,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,factor_mensual_raw,factor_mensual,es_mes_cosecha,produccion_mensual_t,area_mensual_ha,perdida_real_pct,perdida_real_mensual_pct,evento_perdida_real,precio_ico_usd_ton,precio_productor_usd_ton
0,2000-01-01,Cundinamarca,1,2000,58.122070,16.040529,80.992858,1030.010331,87.170530,969.042186,881.660989,7.203958,NaN,NaN,NaN,NaN,0.045104,NaN,NaN,1895.307361,1086.421734,14.454071,10.320921,185.295099,NaN,NaN,-15.291502,NaN,-6.983227,-15.291502,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-02-01,Cundinamarca,2,2000,102.480240,16.364601,77.921106,1096.139029,4.391722,1015.409971,1010.790421,7.418556,0.275704,0.194953,25.120398,7.237869,0.046012,1.003104,0.286976,1895.307361,1086.421734,14.454071,10.320921,185.295099,-50.827307,-44.414822,-2.640640,-43.957202,25.536483,-32.627589,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-03-01,Cundinamarca,3,2000,126.669177,16.723544,77.873874,1071.270787,24.697898,1135.673349,1110.807486,7.621710,0.504174,0.331648,23.907992,11.393505,0.044515,1.484609,0.373975,1895.307361,1086.421734,14.454071,10.320921,185.295099,-3.017617,-2.038318,2.444149,-7.367417,-10.835857,-0.870595,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-04-01,Cundinamarca,4,2000,146.966135,16.678842,81.240824,1098.622452,11.807117,990.875170,978.952966,7.806545,0.438807,0.307035,23.645491,6.576422,0.048026,1.149813,0.300540,1895.307361,1086.421734,14.454071,10.320921,185.295099,-15.700155,-12.968768,8.392347,-24.035019,-30.175531,-6.758859,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-05-01,Cundinamarca,5,2000,238.109657,16.730210,83.152572,1182.556733,0.000000,891.637309,891.514079,7.702863,0.430630,0.286162,23.446429,8.874900,0.049690,1.030764,0.267016,1895.307361,1086.421734,14.454071,10.320921,185.295099,-16.584008,-16.898203,3.782529,-34.932718,13.446331,-9.899894,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Recorte al periodo con Agronet observado

La base anual de modelado debe construirse solo sobre el periodo donde existe observacion productiva real. Por eso nos quedamos con 2007-2024.


In [4]:
df_model = df[df['anio'].between(2007, 2024)].copy()

print('Shape recortada para modelado anual:', df_model.shape)
print('Periodo:', df_model['fecha'].min(), '->', df_model['fecha'].max())
print('Departamentos:', sorted(df_model['departamento'].unique().tolist()))


Shape recortada para modelado anual: (432, 56)
Periodo: 2007-01-01 00:00:00 -> 2024-12-01 00:00:00
Departamentos: ['Cundinamarca', 'Risaralda']


## Separacion conceptual de variables

Aqui definimos que variables se van a agregar y con que logica general.


In [5]:
vars_sum_annual = ['precipitation']

vars_mean_annual = [
    'temp_aire_C', 'humedad_relativa_pct', 'soil', 'def', 'pet', 'aet', 'GDD_cafe',
    'NDVI', 'EVI', 'LST_Day_1km', 'LST_Night_1km', 'Gpp', 'Lai_500m', 'Fpar_500m',
    'NDVI_anomalia_pct', 'EVI_anomalia_pct', 'Gpp_anomalia_pct', 'Lai_500m_anomalia_pct', 'precipitation_anomalia_pct',
    'indice_perdida',
]

terrain_cols = ['elevacion_media_m', 'elevacion_std_m', 'pendiente_media', 'pendiente_std', 'aspecto_medio']

agronet_static_cols = [
    'n_municipios',
    'area_cosechada_ha', 'area_sembrada_ha',
    'produccion_t_original', 'rendimiento_t_ha_original',
    'produccion_t', 'rendimiento_t_ha',
    'correccion_aplicada', 'motivo_correccion',
    'delta_produccion_t', 'delta_rendimiento_t_ha',
    'rendimiento_medio_municipal_reportado',
    'dif_rendimiento_calculado_vs_reportado',
    'rendimiento_medio_t_ha', 'produccion_media_t',
    'perdida_real_pct', 'evento_perdida_real',
    'precio_ico_usd_ton', 'precio_productor_usd_ton',
]

vars_sum_annual = [c for c in vars_sum_annual if c in df_model.columns]
vars_mean_annual = [c for c in vars_mean_annual if c in df_model.columns]
terrain_cols = [c for c in terrain_cols if c in df_model.columns]
agronet_static_cols = [c for c in agronet_static_cols if c in df_model.columns]

print('Variables suma anual:', vars_sum_annual)
print('Variables media anual:', vars_mean_annual)
print('Variables terreno:', terrain_cols)
print('Variables Agronet estaticas:', agronet_static_cols)


Variables suma anual: ['precipitation']
Variables media anual: ['temp_aire_C', 'humedad_relativa_pct', 'soil', 'def', 'pet', 'aet', 'GDD_cafe', 'NDVI', 'EVI', 'LST_Day_1km', 'LST_Night_1km', 'Gpp', 'Lai_500m', 'Fpar_500m', 'NDVI_anomalia_pct', 'EVI_anomalia_pct', 'Gpp_anomalia_pct', 'Lai_500m_anomalia_pct', 'precipitation_anomalia_pct', 'indice_perdida']
Variables terreno: ['elevacion_media_m', 'elevacion_std_m', 'pendiente_media', 'pendiente_std', 'aspecto_medio']
Variables Agronet estaticas: ['n_municipios', 'area_cosechada_ha', 'area_sembrada_ha', 'produccion_t_original', 'rendimiento_t_ha_original', 'produccion_t', 'rendimiento_t_ha', 'correccion_aplicada', 'motivo_correccion', 'delta_produccion_t', 'delta_rendimiento_t_ha', 'rendimiento_medio_municipal_reportado', 'dif_rendimiento_calculado_vs_reportado', 'rendimiento_medio_t_ha', 'produccion_media_t', 'perdida_real_pct', 'evento_perdida_real', 'precio_ico_usd_ton', 'precio_productor_usd_ton']


## Agregados anuales y de cosecha

Se construyen dos familias de features:

- agregados sobre todos los meses del año
- agregados solo sobre meses de cosecha

Esto permite luego comparar si la señal relevante del modelo vive en el año completo o en el subconjunto operativo de cosecha.


In [6]:
group_keys = ['departamento', 'anio']

annual_sum = df_model.groupby(group_keys)[vars_sum_annual].sum(min_count=1).add_suffix('_annual_sum')
annual_mean = df_model.groupby(group_keys)[vars_mean_annual].mean().add_suffix('_annual_mean')
terrain_first = df_model.groupby(group_keys)[terrain_cols].first()

df_cosecha = df_model[df_model['es_mes_cosecha'] == 1].copy()
cosecha_sum = df_cosecha.groupby(group_keys)[vars_sum_annual].sum(min_count=1).add_suffix('_cosecha_sum')
cosecha_mean = df_cosecha.groupby(group_keys)[vars_mean_annual].mean().add_suffix('_cosecha_mean')

base_features = pd.concat([annual_sum, annual_mean, cosecha_sum, cosecha_mean, terrain_first], axis=1).reset_index()

print('Shape base de features agregadas:', base_features.shape)
display(base_features.head(6))


Shape base de features agregadas: (36, 49)


,departamento,anio,precipitation_annual_sum,temp_aire_C_annual_mean,humedad_relativa_pct_annual_mean,soil_annual_mean,def_annual_mean,pet_annual_mean,aet_annual_mean,GDD_cafe_annual_mean,NDVI_annual_mean,EVI_annual_mean,LST_Day_1km_annual_mean,LST_Night_1km_annual_mean,Gpp_annual_mean,Lai_500m_annual_mean,Fpar_500m_annual_mean,NDVI_anomalia_pct_annual_mean,EVI_anomalia_pct_annual_mean,Gpp_anomalia_pct_annual_mean,Lai_500m_anomalia_pct_annual_mean,precipitation_anomalia_pct_annual_mean,indice_perdida_annual_mean,precipitation_cosecha_sum,temp_aire_C_cosecha_mean,humedad_relativa_pct_cosecha_mean,soil_cosecha_mean,def_cosecha_mean,pet_cosecha_mean,aet_cosecha_mean,GDD_cafe_cosecha_mean,NDVI_cosecha_mean,EVI_cosecha_mean,LST_Day_1km_cosecha_mean,LST_Night_1km_cosecha_mean,Gpp_cosecha_mean,Lai_500m_cosecha_mean,Fpar_500m_cosecha_mean,NDVI_anomalia_pct_cosecha_mean,EVI_anomalia_pct_cosecha_mean,Gpp_anomalia_pct_cosecha_mean,Lai_500m_anomalia_pct_cosecha_mean,precipitation_anomalia_pct_cosecha_mean,indice_perdida_cosecha_mean,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio
0,Cundinamarca,2007,1687.793891,17.049639,78.108544,1041.084581,84.362900,914.525073,830.046049,7.766690,0.559274,0.358158,23.882376,11.317330,0.047687,1.857250,0.443968,-0.195160,-1.670301,-0.785513,2.800224,-6.275240,-0.883658,1098.347069,16.752371,82.927817,1187.465469,0.673783,836.745929,835.960358,7.622968,0.553130,0.358165,22.570806,11.000249,0.045257,1.793670,0.428554,-1.836137,-2.961987,-2.989640,0.026393,3.079408,-2.595921,1895.307361,1086.421734,14.454071,10.320921,185.295099
1,Cundinamarca,2008,1906.311791,16.676312,80.937549,1116.492910,24.495232,879.463229,854.818542,7.530276,0.549232,0.359143,23.041187,10.537202,0.046001,1.687877,0.420868,-2.102305,-1.553013,-4.110201,-7.360825,9.353536,-2.588506,1158.837926,16.641577,83.542475,1189.154816,0.285094,828.733359,828.321539,7.464702,0.547475,0.353955,22.929064,9.986722,0.044084,1.728891,0.426625,-3.050697,-4.335153,-5.295259,-5.319884,7.398212,-4.227036,1895.307361,1086.421734,14.454071,10.320921,185.295099
2,Cundinamarca,2009,1588.434373,17.222820,78.467313,1124.390811,29.993428,939.235693,909.166882,8.070870,0.554271,0.361746,23.927916,12.172007,0.048055,1.796530,0.435888,-0.992592,-0.657007,0.398230,-0.680473,-4.401427,-0.417123,888.102849,17.276536,79.594043,1163.311099,16.596303,919.117149,902.484833,8.187371,0.569247,0.371836,23.970807,11.922794,0.047742,1.940012,0.463285,0.868999,0.629689,2.547245,6.786979,-16.970570,1.348644,1895.307361,1086.421734,14.454071,10.320921,185.295099
3,Cundinamarca,2010,1990.662586,17.267404,80.521146,993.511513,92.000587,923.123544,831.051482,8.299793,0.567236,0.367038,23.982538,11.464887,0.045272,1.794000,0.437380,1.452131,0.894429,-5.701729,-0.832945,11.336853,-1.118390,1289.932945,16.860969,85.508868,1191.991180,0.000000,843.301483,843.237074,8.074171,0.552335,0.366027,23.063761,10.930399,0.044351,1.737107,0.423899,-1.659950,-0.673494,-4.852863,-3.053837,24.026872,-2.395436,1895.307361,1086.421734,14.454071,10.320921,185.295099
4,Cundinamarca,2011,2116.188010,16.659077,82.688585,1154.752902,13.848185,893.893232,879.972841,8.022986,0.565623,0.372965,22.787445,10.909606,0.046029,1.770150,0.435307,1.022006,2.514288,-4.194750,-2.265908,20.395981,-0.219485,1356.913769,16.564442,86.047057,1194.073206,0.029760,816.326832,816.246022,7.988944,0.552538,0.365921,22.235671,9.874574,0.042161,1.677779,0.416596,-1.939740,-0.812538,-9.515386,-7.185137,26.533556,-4.089221,1895.307361,1086.421734,14.454071,10.320921,185.295099
5,Cundinamarca,2012,1705.598743,16.881437,79.316418,1052.323829,55.266973,949.000746,893.688984,8.249781,0.565009,0.366467,23.645984,11.805137,0.047432,1.806101,0.442161,0.619430,0.447281,-0.994412,-0.637465,-0.959082,0.024100,1014.189747,16.880681,81.385721,1156.021883,12.834680,928.237022,915.366605,8.158683,0.577597,0.378190,23.340023,11.624666,0.046615,1.935282,0.463161,2.125403,2.178716,-0.063115,6.233281,-6.084329,1.413668,

## Capa anual observada de Agronet y negocio

Las variables anuales observadas de Agronet y precio se repiten mensualmente en la base integrada. Aqui las colapsamos a una sola fila por `departamento-anio` usando el primer valor no nulo del grupo.


In [7]:
def first_non_null(series):
    non_null = series.dropna()
    return non_null.iloc[0] if not non_null.empty else np.nan


agronet_annual = (
    df_model.groupby(group_keys)[agronet_static_cols]
    .agg(first_non_null)
    .reset_index()
)

print('Shape capa anual Agronet-negocio:', agronet_annual.shape)
display(agronet_annual.head(6))


Shape capa anual Agronet-negocio: (36, 21)


,departamento,anio,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real,precio_ico_usd_ton,precio_productor_usd_ton
0,Cundinamarca,2007,65.0,43017.30,48195.69,33729.143730,0.784083,33729.143730,0.784083,0.0,NaN,0.000000,0.000000,0.629565,0.154518,0.925118,30017.744086,-15.245048,1.0,2750.0,2145.0
1,Cundinamarca,2008,66.0,43633.35,48989.09,78254.745626,1.793462,35732.355721,0.818923,1.0,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,-42522.389905,-0.974539,1.908602,-0.115141,0.925118,30017.744086,-11.479062,0.0,2900.0,2262.0
2,Cundinamarca,2009,69.0,43475.84,48581.30,37118.057049,0.853763,37118.057049,0.853763,0.0,NaN,0.000000,0.000000,0.751595,0.102168,0.925118,30017.744086,-7.713077,0.0,2600.0,2028.0
3,Cundinamarca,2010,69.0,44264.16,49357.78,37214.800000,0.840743,37214.800000,0.840743,0.0,NaN,0.000000,0.000000,0.783393,0.057350,0.925118,30017.744086,-9.120406,0.0,3500.0,2730.0
4,Cundinamarca,2011,69.0,37478.87,46123.78,32780.348700,0.874635,32780.348700,0.874635,0.0,NaN,0.000000,0.000000,0.854516,0.020120,0.925118,30017.744086,-5.456866,0.0,5900.0,4602.0
5,Cundinamarca,2012,68.0,37175.06,45269.86,30786.406900,0.828147,30786.406900,0.828147,0.0,NaN,0.000000,0.000000,0.732413,0.095734,0.925118,30017.744086,-10.482027,0.0,4000.0,3120.0


## Integracion de features anuales con Agronet anual

Aqui se construye la base anual consolidada, uniendo la capa observada de Agronet con los agregados climaticos y satelitales.


In [8]:
df_anual = agronet_annual.merge(base_features, on=group_keys, how='left')

print('Shape base anual consolidada:', df_anual.shape)
display(df_anual.head(6))


Shape base anual consolidada: (36, 68)


,departamento,anio,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real,precio_ico_usd_ton,precio_productor_usd_ton,precipitation_annual_sum,temp_aire_C_annual_mean,humedad_relativa_pct_annual_mean,soil_annual_mean,def_annual_mean,pet_annual_mean,aet_annual_mean,GDD_cafe_annual_mean,NDVI_annual_mean,EVI_annual_mean,LST_Day_1km_annual_mean,LST_Night_1km_annual_mean,Gpp_annual_mean,Lai_500m_annual_mean,Fpar_500m_annual_mean,NDVI_anomalia_pct_annual_mean,EVI_anomalia_pct_annual_mean,Gpp_anomalia_pct_annual_mean,Lai_500m_anomalia_pct_annual_mean,precipitation_anomalia_pct_annual_mean,indice_perdida_annual_mean,precipitation_cosecha_sum,temp_aire_C_cosecha_mean,humedad_relativa_pct_cosecha_mean,soil_cosecha_mean,def_cosecha_mean,pet_cosecha_mean,aet_cosecha_mean,GDD_cafe_cosecha_mean,NDVI_cosecha_mean,EVI_cosecha_mean,LST_Day_1km_cosecha_mean,LST_Night_1km_cosecha_mean,Gpp_cosecha_mean,Lai_500m_cosecha_mean,Fpar_500m_cosecha_mean,NDVI_anomalia_pct_cosecha_mean,EVI_anomalia_pct_cosecha_mean,Gpp_anomalia_pct_cosecha_mean,Lai_500m_anomalia_pct_cosecha_mean,precipitation_anomalia_pct_cosecha_mean,indice_perdida_cosecha_mean,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio
0,Cundinamarca,2007,65.0,43017.30,48195.69,33729.143730,0.784083,33729.143730,0.784083,0.0,NaN,0.000000,0.000000,0.629565,0.154518,0.925118,30017.744086,-15.245048,1.0,2750.0,2145.0,1687.793891,17.049639,78.108544,1041.084581,84.362900,914.525073,830.046049,7.766690,0.559274,0.358158,23.882376,11.317330,0.047687,1.857250,0.443968,-0.195160,-1.670301,-0.785513,2.800224,-6.275240,-0.883658,1098.347069,16.752371,82.927817,1187.465469,0.673783,836.745929,835.960358,7.622968,0.553130,0.358165,22.570806,11.000249,0.045257,1.793670,0.428554,-1.836137,-2.961987,-2.989640,0.026393,3.079408,-2.595921,1895.307361,1086.421734,14.454071,10.320921,185.295099
1,Cundinamarca,2008,66.0,43633.35,48989.09,78254.745626,1.793462,35732.355721,0.818923,1.0,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,-42522.389905,-0.974539,1.908602,-0.115141,0.925118,30017.744086,-11.479062,0.0,2900.0,2262.0,1906.311791,16.676312,80.937549,1116.492910,24.495232,879.463229,854.818542,7.530276,0.549232,0.359143,23.041187,10.537202,0.046001,1.687877,0.420868,-2.102305,-1.553013,-4.110201,-7.360825,9.353536,-2.588506,1158.837926,16.641577,83.542475,1189.154816,0.285094,828.733359,828.321539,7.464702,0.547475,0.353955,22.929064,9.986722,0.044084,1.728891,0.426625,-3.050697,-4.335153,-5.295259,-5.319884,7.398212,-4.227036,1895.307361,1086.421734,14.454071,10.320921,185.295099
2,Cundinamarca,2009,69.0,43475.84,48581.30,37118.057049,0.853763,37118.057049,0.853763,0.0,NaN,0.000000,0.000000,0.751595,0.102168,0.925118,30017.744086,-7.713077,0.0,2600.0,2028.0,1588.434373,17.222820,78.467313,1124.390811,29.993428,939.235693,909.166882,8.070870,0.554271,0.361746,23.927916,12.172007,0.048055,1.796530,0.435888,-0.992592,-0.657007,0.398230,-0.680473,-4.401427,-0.417123,888.102849,17.276536,79.594043,1163.311099,16.596303,919.117149,902.484833,8.187371,0.569247,0.371836,23.970807,11.922794,0.047742,1.940012,0.463285,0.868999,0.629689,2.547245,6.786979,-16.970570,1.348644,1895.307361,1086.421734,14.454071,10.320921,185.295099
3,Cundinamarca,2010,69.0,44264.16,49357.78,37214.800000,0.840743,37214.800000,0.840743,0.0,NaN,0.000000,0.000000,0.783393,0.057350,0.925118,30017.744086,-9.120406,0.0,3500.0,2730.0,1990.662586,17.267404,80.521146,993.511513,92.000587,923.123544,831.051482,8.299793,0.567236,0.367038,23.982538,11.464887,0.045272,1.794000,0.437380,1.452131,0.894429,-5.701729,-0.832945,11.336853,-1.118390,1289.932945,16.860969,85.508868,1191.991180,0.000000,843

## Construccion de referencias de rendimiento

Se construyen tres columnas de referencia:

- `rendimiento_referencia_fullsample`: referencia departamental de muestra completa
- `rendimiento_referencia_prev`: promedio de años previos del mismo departamento
- `rendimiento_referencia_prev3`: promedio movil de los tres años previos

La referencia principal del proyecto en esta etapa sera `rendimiento_referencia_fullsample`, porque mantiene consistencia con la logica ya usada en la base integrada y nos permite validar contra `perdida_real_pct`.


In [9]:
df_anual = df_anual.sort_values(['departamento', 'anio']).reset_index(drop=True)

df_anual['rendimiento_referencia_fullsample'] = df_anual['rendimiento_medio_t_ha']

df_anual['rendimiento_referencia_prev'] = (
    df_anual.groupby('departamento')['rendimiento_t_ha']
    .transform(lambda s: s.expanding().mean().shift(1))
)

df_anual['rendimiento_referencia_prev3'] = (
    df_anual.groupby('departamento')['rendimiento_t_ha']
    .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
)

display(
    df_anual[
        ['departamento', 'anio', 'rendimiento_t_ha', 'rendimiento_referencia_fullsample', 'rendimiento_referencia_prev', 'rendimiento_referencia_prev3']
    ].head(10)
)


,departamento,anio,rendimiento_t_ha,rendimiento_referencia_fullsample,rendimiento_referencia_prev,rendimiento_referencia_prev3
0,Cundinamarca,2007,0.784083,0.925118,NaN,NaN
1,Cundinamarca,2008,0.818923,0.925118,0.784083,0.784083
2,Cundinamarca,2009,0.853763,0.925118,0.801503,0.801503
3,Cundinamarca,2010,0.840743,0.925118,0.818923,0.818923
4,Cundinamarca,2011,0.874635,0.925118,0.824378,0.837810
5,Cundinamarca,2012,0.828147,0.925118,0.834430,0.856381
6,Cundinamarca,2013,0.690641,0.925118,0.833382,0.847842
7,Cundinamarca,2014,0.747053,0.925118,0.812991,0.797808
8,Cundinamarca,2015,0.913894,0.925118,0.804749,0.755280
9,Cundinamarca,2016,0.945781,0.925118,0.816876,0.783863


## Construccion operativa del target anual

Se crean dos versiones del target:

- `perdida_rendimiento_anual_pct`: contra referencia full sample
- `perdida_rendimiento_anual_pct_prev`: contra referencia de años previos

Tambien se genera la variable binaria `evento_perdida_anual` usando un umbral de `-15%`, heredado de la logica previa del proyecto. Ese umbral podra revisarse mas adelante en la capa prescriptiva.


In [10]:
UMBRAL_EVENTO = -15

df_anual['perdida_rendimiento_anual_pct'] = (
    (df_anual['rendimiento_t_ha'] - df_anual['rendimiento_referencia_fullsample']) /
    df_anual['rendimiento_referencia_fullsample'] * 100
)

df_anual['perdida_rendimiento_anual_pct_prev'] = (
    (df_anual['rendimiento_t_ha'] - df_anual['rendimiento_referencia_prev']) /
    df_anual['rendimiento_referencia_prev'] * 100
)

df_anual['perdida_rendimiento_anual_pct_prev3'] = (
    (df_anual['rendimiento_t_ha'] - df_anual['rendimiento_referencia_prev3']) /
    df_anual['rendimiento_referencia_prev3'] * 100
)

df_anual['evento_perdida_anual'] = (df_anual['perdida_rendimiento_anual_pct'] < UMBRAL_EVENTO).astype(int)

display(
    df_anual[
        ['departamento', 'anio', 'rendimiento_t_ha', 'perdida_rendimiento_anual_pct', 'perdida_rendimiento_anual_pct_prev', 'perdida_rendimiento_anual_pct_prev3', 'evento_perdida_anual']
    ].head(12)
)


,departamento,anio,rendimiento_t_ha,perdida_rendimiento_anual_pct,perdida_rendimiento_anual_pct_prev,perdida_rendimiento_anual_pct_prev3,evento_perdida_anual
0,Cundinamarca,2007,0.784083,-15.245048,NaN,NaN,1
1,Cundinamarca,2008,0.818923,-11.479062,4.443381,4.443381,0
2,Cundinamarca,2009,0.853763,-7.713077,6.520213,6.520213,0
3,Cundinamarca,2010,0.840743,-9.120406,2.664518,2.664518,0
4,Cundinamarca,2011,0.874635,-5.456866,6.096393,4.395473,0
5,Cundinamarca,2012,0.828147,-10.482027,-0.752945,-3.296872,0
6,Cundinamarca,2013,0.690641,-25.345584,-17.127922,-18.541254,1
7,Cundinamarca,2014,0.747053,-19.247836,-8.110565,-6.361826,1
8,Cundinamarca,2015,0.913894,-1.213192,13.562724,21.000700,0
9,Cundinamarca,2016,0.945781,2.233614,15.780309,20.656499,0


## Validacion de consistencia contra la variable historica del proyecto

Como la referencia `fullsample` es equivalente a la media departamental ya usada en la base integrada, la nueva variable `perdida_rendimiento_anual_pct` deberia coincidir con `perdida_real_pct` salvo pequeñas diferencias numericas de redondeo.


In [11]:
df_anual['diff_target_vs_perdida_real_pct'] = df_anual['perdida_rendimiento_anual_pct'] - df_anual['perdida_real_pct']

validacion_target = {
    'max_abs_diff': float(df_anual['diff_target_vs_perdida_real_pct'].abs().max()),
    'mean_abs_diff': float(df_anual['diff_target_vs_perdida_real_pct'].abs().mean()),
}

print(json.dumps(validacion_target, indent=2, ensure_ascii=False))


{
  "max_abs_diff": 1.9539925233402755e-14,
  "mean_abs_diff": 6.315935428978668e-15
}


## Reordenamiento de columnas finales

En esta etapa dejamos primero identificacion y target, luego la capa productiva observada y finalmente los features agregados.


In [12]:
front_cols = [
    'departamento', 'anio',
    'rendimiento_t_ha', 'produccion_t', 'area_cosechada_ha', 'area_sembrada_ha',
    'rendimiento_referencia_fullsample', 'rendimiento_referencia_prev', 'rendimiento_referencia_prev3',
    'perdida_rendimiento_anual_pct', 'perdida_rendimiento_anual_pct_prev', 'perdida_rendimiento_anual_pct_prev3',
    'evento_perdida_anual',
    'perdida_real_pct', 'evento_perdida_real', 'diff_target_vs_perdida_real_pct',
    'precio_ico_usd_ton', 'precio_productor_usd_ton',
    'correccion_aplicada', 'motivo_correccion',
]

front_cols = [c for c in front_cols if c in df_anual.columns]
remaining_cols = [c for c in df_anual.columns if c not in front_cols]
df_modelado_anual = df_anual[front_cols + remaining_cols].copy()

print('Shape final dataset modelado anual:', df_modelado_anual.shape)
display(df_modelado_anual.head(6))


Shape final dataset modelado anual: (36, 76)


,departamento,anio,rendimiento_t_ha,produccion_t,area_cosechada_ha,area_sembrada_ha,rendimiento_referencia_fullsample,rendimiento_referencia_prev,rendimiento_referencia_prev3,perdida_rendimiento_anual_pct,perdida_rendimiento_anual_pct_prev,perdida_rendimiento_anual_pct_prev3,evento_perdida_anual,perdida_real_pct,evento_perdida_real,diff_target_vs_perdida_real_pct,precio_ico_usd_ton,precio_productor_usd_ton,correccion_aplicada,motivo_correccion,n_municipios,produccion_t_original,rendimiento_t_ha_original,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,precipitation_annual_sum,temp_aire_C_annual_mean,humedad_relativa_pct_annual_mean,soil_annual_mean,def_annual_mean,pet_annual_mean,aet_annual_mean,GDD_cafe_annual_mean,NDVI_annual_mean,EVI_annual_mean,LST_Day_1km_annual_mean,LST_Night_1km_annual_mean,Gpp_annual_mean,Lai_500m_annual_mean,Fpar_500m_annual_mean,NDVI_anomalia_pct_annual_mean,EVI_anomalia_pct_annual_mean,Gpp_anomalia_pct_annual_mean,Lai_500m_anomalia_pct_annual_mean,precipitation_anomalia_pct_annual_mean,indice_perdida_annual_mean,precipitation_cosecha_sum,temp_aire_C_cosecha_mean,humedad_relativa_pct_cosecha_mean,soil_cosecha_mean,def_cosecha_mean,pet_cosecha_mean,aet_cosecha_mean,GDD_cafe_cosecha_mean,NDVI_cosecha_mean,EVI_cosecha_mean,LST_Day_1km_cosecha_mean,LST_Night_1km_cosecha_mean,Gpp_cosecha_mean,Lai_500m_cosecha_mean,Fpar_500m_cosecha_mean,NDVI_anomalia_pct_cosecha_mean,EVI_anomalia_pct_cosecha_mean,Gpp_anomalia_pct_cosecha_mean,Lai_500m_anomalia_pct_cosecha_mean,precipitation_anomalia_pct_cosecha_mean,indice_perdida_cosecha_mean,elevacion_media_m,elevacion_std_m,pendiente_media,pendiente_std,aspecto_medio
0,Cundinamarca,2007,0.784083,33729.143730,43017.30,48195.69,0.925118,NaN,NaN,-15.245048,NaN,NaN,1,-15.245048,1.0,8.881784e-15,2750.0,2145.0,0.0,NaN,65.0,33729.143730,0.784083,0.000000,0.000000,0.629565,0.154518,0.925118,30017.744086,1687.793891,17.049639,78.108544,1041.084581,84.362900,914.525073,830.046049,7.766690,0.559274,0.358158,23.882376,11.317330,0.047687,1.857250,0.443968,-0.195160,-1.670301,-0.785513,2.800224,-6.275240,-0.883658,1098.347069,16.752371,82.927817,1187.465469,0.673783,836.745929,835.960358,7.622968,0.553130,0.358165,22.570806,11.000249,0.045257,1.793670,0.428554,-1.836137,-2.961987,-2.989640,0.026393,3.079408,-2.595921,1895.307361,1086.421734,14.454071,10.320921,185.295099
1,Cundinamarca,2008,0.818923,35732.355721,43633.35,48989.09,0.925118,0.784083,0.784083,-11.479062,4.443381,4.443381,0,-11.479062,0.0,1.243450e-14,2900.0,2262.0,1.0,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,66.0,78254.745626,1.793462,-42522.389905,-0.974539,1.908602,-0.115141,0.925118,30017.744086,1906.311791,16.676312,80.937549,1116.492910,24.495232,879.463229,854.818542,7.530276,0.549232,0.359143,23.041187,10.537202,0.046001,1.687877,0.420868,-2.102305,-1.553013,-4.110201,-7.360825,9.353536,-2.588506,1158.837926,16.641577,83.542475,1189.154816,0.285094,828.733359,828.321539,7.464702,0.547475,0.353955,22.929064,9.986722,0.044084,1.728891,0.426625,-3.050697,-4.335153,-5.295259,-5.319884,7.398212,-4.227036,1895.307361,1086.421734,14.454071,10.320921,185.295099
2,Cundinamarca,2009,0.853763,37118.057049,43475.84,48581.30,0.925118,0.801503,0.801503,-7.713077,6.520213,6.520213,0,-7.713077,0.0,1.065814e-14,2600.0,2028.0,0.0,NaN,69.0,37118.057049,0.853763,0.000000,0.000000,0.751595,0.102168,0.925118,30017.744086,1588.434373,17.222820,78.467313,1124.390811,29.993428,939.235693,909.166882,8.070870,0.554271,0.361746,23.927916,12.172007,0.048055,1.796530,0.435888,-0.992592,-0.657007,0.398230,-0.680473,-4.401427,-0.417123,888.102849,17.276536,79.594043,1163.311099,16.596303,919.117149,902.484833,8.187371,0.569247,0.371836,23.970807,11.922794,0.047742,1.940012,0.463285,0.868999,0.629689,2.547245,6.786979,-16.970570,1.348644,1895.307361,1086.421734,14.454071,10.320921,185.295099
3,Cundi

## Validaciones finales antes de exportar

Estas validaciones confirman que el dataset final esta listo para pasar a EDA y diseño experimental.


In [13]:
assert df_modelado_anual.duplicated(subset=['departamento', 'anio']).sum() == 0, 'Hay duplicados por departamento-anio.'
assert set(df_modelado_anual['departamento'].unique()) == {'Cundinamarca', 'Risaralda'}, 'Departamentos inesperados en la base anual.'
assert df_modelado_anual['anio'].min() == 2007, 'El anio minimo esperado es 2007.'
assert df_modelado_anual['anio'].max() == 2024, 'El anio maximo esperado es 2024.'
assert len(df_modelado_anual) == 36, 'Se esperaban 36 filas en la base anual de modelado.'
assert df_modelado_anual['rendimiento_t_ha'].notna().all(), 'Hay rendimientos nulos.'
assert df_modelado_anual['perdida_rendimiento_anual_pct'].notna().all(), 'El target principal no debe quedar nulo.'
assert df_modelado_anual['diff_target_vs_perdida_real_pct'].abs().max() < 1e-9, 'El target principal no coincide con la variable anual previa del proyecto.'
assert df_modelado_anual['correccion_aplicada'].sum() == 1, 'Se esperaba una unica correccion de Agronet marcada.'

print('Validaciones finales superadas correctamente.')


Validaciones finales superadas correctamente.


## Exportacion del dataset anual de modelado

La salida se guarda con separador `;` para mantener consistencia con las otras bases procesadas del proyecto.


In [14]:
df_modelado_anual.to_csv(OUTPUT_PATH, index=False, sep=';')
print(f'Archivo exportado en: {OUTPUT_PATH}')


Archivo exportado en: G:\.shortcut-targets-by-id\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\Proyecto de Analytics\PROYECTO_V2\BASE_DE_DATOS\FINALES\dataset_modelado_anual.csv


## Resumen ejecutivo

Este bloque deja una lectura corta del resultado y conecta con la siguiente fase del trabajo.


In [15]:
resumen = {
    'archivo_salida': str(OUTPUT_PATH),
    'shape_final': df_modelado_anual.shape,
    'periodo': f"{int(df_modelado_anual['anio'].min())}-{int(df_modelado_anual['anio'].max())}",
    'departamentos': sorted(df_modelado_anual['departamento'].unique().tolist()),
    'n_eventos_perdida_anual': int(df_modelado_anual['evento_perdida_anual'].sum()),
    'target_consistente_con_perdida_real_pct': bool(df_modelado_anual['diff_target_vs_perdida_real_pct'].abs().max() < 1e-9),
    'siguiente_paso': 'usar este dataset para EDA anual, seleccion exploratoria de variables y definicion del split temporal de modelado',
}

print(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))


{
  "archivo_salida": "G:\\.shortcut-targets-by-id\\1LEoqVVNgeO19T3qHr1j6Gf33gUoTUgDI\\Proyecto de Analytics\\PROYECTO_V2\\BASE_DE_DATOS\\FINALES\\dataset_modelado_anual.csv",
  "shape_final": [
    36,
    76
  ],
  "periodo": "2007-2024",
  "departamentos": [
    "Cundinamarca",
    "Risaralda"
  ],
  "n_eventos_perdida_anual": 7,
  "target_consistente_con_perdida_real_pct": true,
  "siguiente_paso": "usar este dataset para EDA anual, seleccion exploratoria de variables y definicion del split temporal de modelado"
}


## Siguiente notebook recomendado

El siguiente paso natural es construir `06_limpieza_y_qc_final.ipynb` si quieres cerrar una capa final de validacion transversal, o pasar directamente a `07_eda_base_anual.ipynb` para explorar esta nueva base anual de modelado.
